In [1]:
import os
import pickle
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
from torch import tensor
import numpy as np
import csv


input_size = 75
hidden_layers = [256, 256,256]
output_size = 4  # Updated output_size to match the number of classes
dropout_probs= [0.0, 0.0,0.0]
lr =  1e-4
batch_size = 512


print(input_size, hidden_layers, output_size, dropout_probs)
  

directory = 'Figures/['
for a in range(len(hidden_layers)):
    directory += str(hidden_layers[a])+','
    if a == len(hidden_layers)-1:
        directory = directory[:-1]
        directory += ']'
if not os.path.isdir(directory):
    os.makedirs(directory)
out = directory
# Define a custom PyTorch Dataset
class CustomDataset(Dataset):
    def __init__(self, data_dir):
        self.data_dir = data_dir
        # Pre-allocate lists to hold raw numpy data
        all_energies = []
        all_intensities = []
        all_labels = []

        for filename in os.listdir(self.data_dir):
            if filename.endswith("_energy.pkl"):
                label_str = filename.split("_")[0][1:] 
                label = int(label_str)

                energy_path = os.path.join(self.data_dir, filename)
                intensity_path = os.path.join(self.data_dir, f"P{label_str}_intensity.pkl")

                if os.path.exists(intensity_path):
                    with open(energy_path, "rb") as f:
                        e_data = np.array(pickle.load(f))
                    with open(intensity_path, "rb") as f:
                        i_data = np.array(pickle.load(f))

                    if e_data.shape == i_data.shape:
                        all_energies.append(e_data)
                        all_intensities.append(i_data)
                        # Create a label array matching the number of samples in these files
                        all_labels.append(np.full(e_data.shape[0], label))

        # --- THE OPTIMIZATION ---
        # Concatenate everything into massive Numpy arrays first
        full_energy = np.concatenate(all_energies, axis=0)
        full_intensity = np.concatenate(all_intensities, axis=0)
        full_labels = np.concatenate(all_labels, axis=0)

        # Convert the entire dataset to Tensors ONCE. 
        self.energy_tensors = torch.from_numpy(full_energy).float()
        self.intensity_tensors = torch.from_numpy(full_intensity).float()
        self.labels = torch.from_numpy(full_labels).long()

    def __len__(self):
        return self.labels.size(0)

    def __getitem__(self, idx):
        # Indexing a large tensor is much faster than creating a new one
        # Returns shape (2, 45)
        input_tensor = torch.stack((self.energy_tensors[idx], self.intensity_tensors[idx]), dim=0)
        return input_tensor, self.labels[idx]

75 [256, 256, 256] 4 [0.0, 0.0, 0.0]


In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class SimpleDNN(nn.Module):
    def __init__(self, output_size=output_size, hidden_layers=hidden_layers):
        super(SimpleDNN, self).__init__()
        
        # Convolutional Layers
        self.conv1 = nn.Conv1d(2, 8, kernel_size=5, padding=2)
        self.conv2 = nn.Conv1d(8, 16, kernel_size=5, padding=2)
        self.conv3 = nn.Conv1d(16, 32, kernel_size=5, padding=2)
        
        # Adaptive Max Pool: Forces the sequence length to be exactly 

        # Now, flatten_size is ALWAYS equal to the number of out_channels (32)
        # because the sequence length (45) was squashed to 1.
        self.flatten_size = 32 * input_size

        self.fc1 = nn.Linear(self.flatten_size, hidden_layers[0])
        self.fc2 = nn.Linear(hidden_layers[0], hidden_layers[1])
        self.fc3 = nn.Linear(hidden_layers[1], hidden_layers[2])
        self.fc4 = nn.Linear(hidden_layers[2], output_size)

        self.dropout = nn.Dropout(p=0.15)

    def forward(self, x):
        # Convolution layers
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = F.relu(self.conv3(x))
        
        # Flatten (removes the last dimension)
        x = x.view(x.size(0), -1) # Shape: (Batch, 32)
    
        # Fully Connected Layers
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = F.relu(self.fc2(x))
        x = self.dropout(x)
        x = F.relu(self.fc3(x))
        x = self.dropout(x)
        
        return self.fc4(x)

In [3]:
import time
import os
import csv
import torch

def train_gpu_resident(model, train_x, train_y, val_x, val_y, criterion, optimizer, scheduler, 
                       num_epochs, batch_size, checkpoint_dir=directory+"/checkpoints", metrics_file=directory+"/training_metrics.csv"):
    
    # --- Configuration ---
    best_val_loss = float('inf')
    #early_stop_patience = 80  # Stop if no improvement for 50 epochs
    epochs_no_improve = 0
    min_delta = 0.0020
    num_samples = train_x.size(0)
    
    # Ensure checkpoint directory exists
    os.makedirs(checkpoint_dir, exist_ok=True)

    # Open CSV once and keep it open (matches train_model logic)
    with open(metrics_file, mode="w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["Epoch", "Train Loss", "Train Accuracy", "Val Loss", "Val Accuracy", "LR"])

        for epoch in range(num_epochs):
            start_time = time.time()
            
            # --- Training Phase ---
            model.train()
            indices = torch.randperm(num_samples, device=train_x.device)
            running_loss, correct_train, total_train = 0.0, 0, 0
            
            # Calculate total batches for loss division later
            num_batches = (num_samples + batch_size - 1) // batch_size
            
            for i in range(0, num_samples, batch_size):
                batch_idx = indices[i: i + batch_size]
                inputs, labels = train_x[batch_idx], train_y[batch_idx]
    
                optimizer.zero_grad()
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                loss.backward()
                
                # STABILITY: Gradient Clipping
                #torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0) ### ADDS STABILITY TO TRAINING, adjusting max_norm can provide better stability by avoiding large deviation of weights
                optimizer.step()
                
                # --- TRACKING ACCURACY DATA ---
                running_loss += loss.item()
                _, predicted = torch.max(outputs.data, 1)
                total_train += labels.size(0)
                correct_train += (predicted == labels).sum().item()
    
            # --- FINAL METRIC CALCULATIONS ---
            train_loss = running_loss / num_batches
            train_accuracy = 100 * correct_train / total_train
            vram_used = torch.cuda.memory_reserved() / 1e6 if torch.cuda.is_available() else 0
            
            # --- Validation Phase ---
            model.eval()
            with torch.no_grad():
                val_outputs = model(val_x)
                val_loss = criterion(val_outputs, val_y).item()
                
                _, predicted = torch.max(val_outputs.data, 1)
                total_val = val_y.size(0)
                correct_val = (predicted == val_y).sum().item()
                val_accuracy = 100 * correct_val / total_val
            

            # --- Scheduler & LR Tracking ---
            #scheduler.step(val_loss) #--- Avoiding plateaus is our current priority making sure that we let the machine learn as much as it can. 
                        #Trials have shown that our model is currently not at risk of extreme overfitting 
            current_lr = optimizer.param_groups[0]['lr']
            epoch_duration = time.time() - start_time 

            
            # --- Save metrics to CSV (Matches train_model) ---
            writer.writerow([epoch + 1, train_loss, train_accuracy, val_loss, val_accuracy, current_lr
                            ])
            f.flush()
    
            # --- PRINT STATEMENT ---
            print(f"Epoch [{epoch+1}/{num_epochs}]| Time: {epoch_duration:.2f}s | VRAM: {vram_used:.0f}MB | "
                  f"Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Train Acc: {train_accuracy:.2f}% | Val Acc: {val_accuracy:.2f}% | LR: {current_lr:.6f}| Patience: {epochs_no_improve}"
                 )
    
            # --- Best Model & Early Stopping (Matches train_model) ---
            if val_loss < best_val_loss: 
                if (best_val_loss - val_loss) > min_delta:
                    epochs_no_improve = 0
                else:
                    epochs_no_improve += 1
                
                best_val_loss = val_loss
                torch.save(model.state_dict(), os.path.join(checkpoint_dir, "best_model.pth"))
                print(f"^Best Model Saved")
            else:
                epochs_no_improve += 1
    
            #if epochs_no_improve >= early_stop_patience:
                #print(f"Early stopping triggered at epoch {epoch+1}")
                #break

In [4]:
if __name__ == "__main__":
    data_dir = "Datasets"
    dataset = CustomDataset(data_dir)

    if len(dataset) == 0:
        print(f"No data found.")
    else:
        # 1. Split on CPU as usual
        train_size = int(0.8 * len(dataset))
        val_size = len(dataset) - train_size
        train_ds, val_ds = random_split(dataset, [train_size, val_size], generator=torch.Generator().manual_seed(42))

        # We create this specifically to save it for inference later.
        # It is not used in the training loop below (which uses raw tensors).
        val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False)
        
        os.makedirs(directory, exist_ok=True) # Ensure dir exists
        
        with open(f"{directory}/val_loader.pkl", "wb") as f:
            pickle.dump(val_loader, f)
        print("Validation DataLoader saved to 'val_loader.pkl'.")
        # 2. DEVICE SETUP
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        print(f"Moving dataset to {device}...")

        # 3. MOVE ENTIRE DATASET TO GPU (VRAM)
        # We use a temp loader to extract the data in one large chunk
        full_train_loader = DataLoader(train_ds, batch_size=len(train_ds), num_workers=0)
        full_val_loader = DataLoader(val_ds, batch_size=len(val_ds), num_workers=0)

        # These variables now live on your RTX card
        train_inputs_gpu, train_labels_gpu = next(iter(full_train_loader))
        train_inputs_gpu, train_labels_gpu = train_inputs_gpu.to(device), train_labels_gpu.to(device)

        val_inputs_gpu, val_labels_gpu = next(iter(full_val_loader))
        val_inputs_gpu, val_labels_gpu = val_inputs_gpu.to(device), val_labels_gpu.to(device)

        # 4. INITIALIZE MODEL
        model = SimpleDNN().to(device)
        criterion = nn.CrossEntropyLoss()
        optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, 
            mode='min', 
            factor=0.5, 
            patience=20, 
            min_lr=1e-5, 
            threshold=0.0020, # Use threshold for loss change
            threshold_mode='abs' # Ensures 0.0020 is treated as an absolute value
        )
        # 5. START OPTIMIZED TRAINING
        # We pass the GPU tensors directly instead of a DataLoader
        train_gpu_resident(model, train_inputs_gpu, train_labels_gpu, val_inputs_gpu, val_labels_gpu, 
                           criterion, optimizer, scheduler,
                           num_epochs=1000, batch_size=batch_size)

Validation DataLoader saved to 'val_loader.pkl'.
Moving dataset to cuda...
Epoch [1/1000]| Time: 1.25s | VRAM: 92MB | Train Loss: 2.7225 | Val Loss: 1.3617 | Train Acc: 26.57% | Val Acc: 28.81% | LR: 0.000100| Patience: 0
^Best Model Saved
Epoch [2/1000]| Time: 0.34s | VRAM: 369MB | Train Loss: 1.3565 | Val Loss: 1.3300 | Train Acc: 30.71% | Val Acc: 34.31% | LR: 0.000100| Patience: 0
^Best Model Saved
Epoch [3/1000]| Time: 0.34s | VRAM: 369MB | Train Loss: 1.3383 | Val Loss: 1.3256 | Train Acc: 31.97% | Val Acc: 53.89% | LR: 0.000100| Patience: 0
^Best Model Saved
Epoch [4/1000]| Time: 0.30s | VRAM: 371MB | Train Loss: 1.3319 | Val Loss: 1.3490 | Train Acc: 31.92% | Val Acc: 36.54% | LR: 0.000100| Patience: 0
Epoch [5/1000]| Time: 0.33s | VRAM: 371MB | Train Loss: 1.3474 | Val Loss: 1.3306 | Train Acc: 30.24% | Val Acc: 34.16% | LR: 0.000100| Patience: 1
Epoch [6/1000]| Time: 0.31s | VRAM: 371MB | Train Loss: 1.3305 | Val Loss: 1.3032 | Train Acc: 31.96% | Val Acc: 20.23% | LR: 0.0001

KeyboardInterrupt: 

In [ ]:

content = (
    f"input_size = {input_size}\n"
    f"hidden_layers = {hidden_layers}\n"
    f"output_size = {output_size}\n"
    f"dropout_probs = {dropout_probs}\n"
    f"lr = {lr}\n"
)

# Write to a file
with open(directory+"/model_params.txt", "w") as f:
    f.write(content)

print("File written successfully!")